# Workshop: Objects in Python, part 2

### February 21, 2024

In today's workshop we'll continue talking about classes, this time turning our focus to methods and inheritance. 

## Problem 1: Inheritance with Vehicles

Adapted from PYNative (https://pynative.com/python-object-oriented-programming-oop-exercise/)

Let's suppose we want to represent Vehicles (buses, trains, cars, bikes, etc.).
One option for doing this would be to create a separate class for each possible kind of vehicle.
An alternative is to take advantage of inheritance in Python to avoid doing a lot of extra work.

Every Vehicle will have:

- a name (given by a string; required argument)
- a maximum speed (given by a non-negative float; required argument)
- its mileage (given by a non-negative int; optional argument, defaults to `0`)
- A number of wheels, which is a class attribute, which we will set to 4 for now.

To capture this fact, let's start by creating a class `Vehicle` implementing the above specifications.

  - Write an `__init__` method to include error checking and handle the keyword arguments.
  - Add a `__str__` method so we can print `Vehicle` objects. You may specify this however you like, so long as it is reasonable.
  - Add a method `get_number_of_wheels` that takes no arguments and returns the `number_of_wheels` class attribute.
  
The `Vehicle` class should also have a method `increment_mileage`, which takes a single `int` as its only argument and increases the caller's `mileage` by that amount.

In [18]:
class Vehicle:
    number_of_wheels = 4 # Most vehicles have 4 wheels, at least.
    
    def __init__( self, name, max_speed, mileage=0 ):
        #TODO: error checking: check that name is a string, max_speed nonneg float, etc.
        self.name = name
        self.max_speed = max_speed
        self.mileage = mileage
    
    def get_name( self ):
        return self.name
    
    def __str__( self ):
        return self.get_name()
    
    def get_number_of_wheels( self ):
        return self.number_of_wheels
    
    def increment_mileage( self, inc ):
        self.mileage = self.mileage + inc

In [19]:
v = Vehicle('my car', 10)
print(v)

my car


In [20]:
v

In [21]:
v.get_number_of_wheels()

4

Now, create a class `Bus` that inherits from `Vehicle`.

`Bus` objects should have an additional non-negative integer class attribute `capacity`, describing how many passengers it can fit.

Implement a method `total_fare` that takes a single argument `cost`, the cost of a single rider's fare, and returns the total cost to "rent" the bus-- `cost` multiplied by the `capacity`.

In [36]:
class Bus( Vehicle ):
    capacity = 60
    
    def total_fare( self, cost ):
        if not isinstance( cost, (int,float)):
            raise TypeError('cost should be numeric')
        return self.capacity*cost

In [37]:
# we can instantiate a Bus object even though there's no __init__ in Bus.
# __init__ is inherited from Vehicle
b = Bus( 'citybus', 40.0, 1000 )

In [38]:
b.mileage

1000

In [39]:
b.increment_mileage(10) # Method from the parent class, NOT in Bus

In [40]:
b.mileage

1010

In [41]:
b.total_fare( 10 )

600

In [42]:
b.number_of_wheels

4

Create a class `Bicycle` that inherits from `Vehicle`.
Bikes have two wheels, not four, so don't forget to update the `number_of_wheels` class attribute.

In [44]:
# Naive solution: create Bicycle from scratch
class Bicycle( Vehicle ):
    number_of_wheels = 2 # override the class attribute inherited from Vehicle.

Now, let's create an instance of the `Bicycle` class.

In [46]:
b = Bicycle( 'bikey mcbikeface', 20.0, 1000)

In [47]:
b.number_of_wheels

2

What do you think the next three blocks of code will do?
Make a prediction, then run them to check.

In [48]:
isinstance( b, Bicycle ) # Should be true

True

In [49]:
isinstance( b, Bus ) # Should this be True?

False

In [51]:
isinstance( b, Vehicle ) # Should this be True?

True

## Problem 2: roll-your-own complex numbers

Let's implement our own version of complex numbers. Of course, these are actually available in Python (and in `numpy`, for that matter), but this is a good chance to practice implementing addition and multiplication methods.

Recall that a complex number is a number of the form $a + bi$ where $a,b$ are real numbers, and $i$ is the square root of negative 1. We call $a$ the <i>real part</i> of the complex number and $b$ the <i>imaginary</i> part. For two complex numbers $z_1 = a+bi$ and $z_2 = c + di$, we have:

$$ z_1 + z_2 = (a+c) + (b+d)i. $$
$$ z_1 z_2 = (ac - bd) + (ad + bc)i. $$

Implement a class `MyComplex`, with an `__init__` method that takes two arguments (the real part and imaginary part), and implement all the methods necessary so that we can write things like `z1 + z2` and `z1*z2` when `z1` and `z2` are instances of `MyComplex`, and so that we can write things like `5*z1` or `3.1415*z1`. See https://docs.python.org/3/reference/datamodel.html#emulating-numeric-types and https://docs.python.org/3/library/operator.html for more information.

In addition to the addition and multiplication methods, implement methods `real` and `imag` that let us retrieve the real and imaginary parts, respectively, of a given complex number.

Finally, implement a `__str__` method so that we can print our complex numbers.

In [83]:
class MyComplex:
    
    def __init__( self, a, b ):
        if not isinstance( a, (int,float)):
            raise TypeError('a should be an int or float')
        if not isinstance( b, (int,float)):
            raise TypeError('b should be an int or float')
        self.realpart = a
        self.imagpart = b
        
    def real( self ):
        return self.realpart
    
    def imag( self ):
        return self.imagpart
    
    def __add__( self, other ):
        a = self.real() + other.real()
        b = self.imag() + other.imag()
        return MyComplex( a, b )
    
    def __mul__( self, other ):
        # Type-based dispatch.
        if isinstance( other, MyComplex ):
            a = self.real()
            b = self.imag()
            c = other.real()
            d = other.imag()
            # Now just implement our formula from above.
            res_real = a*c - b*d
            res_imag = a*d + b*c
            return MyComplex( res_real, res_imag )
        elif isinstance( other, (int, float ) ):
            return other*self # We know how to do things like 10*z. This will call __rmul__ below.
        else:
            raise TypeError('Unsupported type for multiplication with MyComplex.')
    
    def __rmul__( self, other ):
        # Assume other is a int/float
        a = other*self.real()
        b = other*self.imag()
        return MyComplex( a,b )
    
    def __str__( self ):
        return '%f + %f i' % (self.real(), self.imag())
    
    def __neg__( self ):
        # Return -z, the negation of this complex number.
        a = -self.real()
        b = -self.imag()
        return MyComplex( a, b )

In [84]:
z1 = MyComplex( 1,1)
z2 = MyComplex( -1, 1)
print(z1 * z2) # Python tries to cal z1.__mul__(z2)

-2.000000 + 0.000000 i


In [86]:
# Python calls 10.__mul__( z1 ); int.__mul__ doesn't know what to do with MyComplex, because we wrote it
# When that fails, Python tries to call z1.__rmul__( 10 )
print(10 * z1)

10.000000 + 10.000000 i


In [88]:
print(z1 * 10) # Python tries to call z1.__mul__( 10 ). Python tries to access 10.real().

10.000000 + 10.000000 i


In [81]:
print( z1 )
print( 10 * z1 )

1.000000 + 1.000000 i
10.000000 + 10.000000 i


In [67]:
z1 + z2 # Called as z1.__add__( z2 )

In [60]:
z1 = MyComplex(1,1)

In [61]:
z1.real() # Correct way to retrieve the attribute z1.realpart

1

In [70]:
z1 = MyComplex( 1,1)
z2 = MyComplex( -1, 1)
z = z1+z2 # This will cause an error unless we implement __add__ in MyComplex
print(z) # Should be 0 + 2i

0.000000 + 2.000000 i


In [71]:
z.real(), z.imag()

(0, 2)

In [72]:
print(z1*z2)

-2.000000 + 0.000000 i


In [69]:
z3 = 5.0 + z
print(z3) # Should be 5 + 12i

TypeError: unsupported operand type(s) for +: 'float' and 'MyComplex'

<b>Bonus challenge:</b> implement methods for computing the reciprocal of an instance of the `MyComplex` class and for dividing one instance by another. See https://en.wikipedia.org/wiki/Complex_number#Reciprocal_and_division for a reminder of how reciprocals and divisions works in the complex plane.
Don't forget to raise an appropriate error in the event of trying to divide by zero!

<b>Bonus challenge:</b> implement a class for quaternions (https://en.wikipedia.org/wiki/Quaternion)